Naive Bayes prédiction

1. Imports des librairies

In [5]:
#%pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score

2. Chargement et création de la variable cible

In [6]:
df = pd.read_csv("BDD_initial\Students Social Media Addiction.csv")

#on créer une variable binaire : 1 si addicted(ici si addicted score >= à 7), 0 sinon
df['Addicted'] = (df['Addicted_Score'] >= 7).astype(int)

<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\lanat\AppData\Local\Temp\ipykernel_17896\3864471762.py:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  df = pd.read_csv("BDD_initial\Students Social Media Addiction.csv")


3. Encodage des variables

In [ ]:
#on définit les features
features_var = ['Gender', 'Academic_Level', 'Most_Used_Platform', 'Affects_Academic_Performance', 'Relationship_Status']
features_num = ['Age', 'Avg_Daily_Usage_Hours', 'Sleep_Hours_Per_Night', 'Mental_Health_Score', 'Conflicts_Over_Social_Media']
#on liste les colonnes encodées
features_var_enc = [col + '_enc' for col in features_var]  

#on convertit chaques variables catégorielle en numérique à l'aide de LabelEncoder
le_dict = {}
for col in features_var:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col])
    le_dict[col] = le

4. Split train/test

On sépare les données en deux groupes:

-80% pour entrainer le modèle
-20% pour tester (données que l'on a jamais vue)

In [ ]:
X = df[features_num+ features_var_enc]
Y = df['Addicted']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

5. Entrainement du modèle

In [13]:
gnb = GaussianNB()
gnb.fit(X_train, Y_train)

Y_pred = gnb.predict(X_test)#classe prédite : 0 ou 1
Y_proba = gnb.predict_proba(X_test)[:, 1]#probabilité d'être addicted

6. Evaluation

In [15]:
acc = accuracy_score(Y_test, Y_pred)
auc = roc_auc_score(Y_test, Y_proba)

print(classification_report(Y_test, Y_pred))

cv_scores = cross_val_score(gnb, X, Y, cv=StratifiedKFold(5), scoring='accuracy')
print(f"CV : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


              precision    recall  f1-score   support

           0       1.00      0.83      0.91        59
           1       0.89      1.00      0.94        82

    accuracy                           0.93       141
   macro avg       0.95      0.92      0.92       141
weighted avg       0.94      0.93      0.93       141

CV : 0.9362 ± 0.0449


Très bonne prédiction de la part du modèle pour les addicted, par contre préfère "sur-prédire" l'addiction plutôt que de la manquer